# TraceletCodeAgent on GAIA

One parameterised notebook covering every Tracelet sweep. Set the config in the next cell and run top to
bottom; output paths are derived from the config, so a config that was already run is served from its
existing `pkl` cache instead of re-running.

To re-run a config that already has cached results, set `RUN_TAG` (e.g. `"_rerun"`) for fresh paths.

In [ ]:
# Pick up edits to the editable-installed smolagents without a kernel restart.
%load_ext autoreload
%autoreload 2

# ===================== experiment configuration =====================
MODEL = "gpt-4o"                 # key of MODELS below
SKELETON_STRATEGY = "direct_prompt"     # "direct_prompt" | "post_process"
N_SAMPLES = 1                          # fill-in candidates per step; 1 = the ablation
USE_TRACELET_PROMPT = True             # tracelet_agent.yaml (template/fill-in protocol) vs default code_agent.yaml
N_QUESTIONS = None                   # GAIA validation questions; None = all 165
MAX_STEPS = 50
RUN_TAG = "_v3"                   # e.g. "_rerun" for fresh output paths
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
ENABLE_THINKING = False  # Together-served open-weight models only

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"
assert SKELETON_STRATEGY in ("direct_prompt", "post_process")

In [25]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = MODEL
provider = MODELS[MODEL]

if provider == "openai":
    model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
    stream_outputs = False
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models (see naiveReAct).
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        client_kwargs={"timeout": 300.0},  # bound a stalled stream instead of hanging forever
    )
    stream_outputs = True  # Qwen via TogetherAI rejects non-streaming requests outright

print(f"{model_name} via {provider} | stream_outputs={stream_outputs}")

gpt-4o via openai | stream_outputs=False


In [26]:
# Standard GAIA tool stack, same as naiveReAct.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

In [27]:
import importlib.resources

import yaml

from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

# None lets CodeAgent fall back to its default code_agent.yaml.
prompt_templates = None
if USE_TRACELET_PROMPT:
    prompt_templates = yaml.safe_load(
        importlib.resources.files("smolagents.prompts").joinpath("tracelet_agent.yaml").read_text()
    )

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=N_SAMPLES,
    skeleton_strategy=SKELETON_STRATEGY,
    stream_outputs=stream_outputs,
    prompt_templates=prompt_templates,
)

In [28]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [29]:
from common_setup import evaluate_agent, question_scorer

STRATEGY_TAG = {"direct_prompt": "direct", "post_process": "postprocess"}[SKELETON_STRATEGY]
N_TAG = "" if N_SAMPLES == 3 else f"_n{N_SAMPLES}"
PROMPT_TAG = "_tp" if USE_TRACELET_PROMPT else ""
RUN_NAME = f"tracelet_{STRATEGY_TAG}{N_TAG}{PROMPT_TAG}{RUN_TAG}_react_{model_name}"

# The first sweep predates this naming scheme; keep it pointed at its original directory.
LEGACY_RUN_NAMES = {("gpt-4o", "post_process", 3): "tracelet_react_gpt-4o"}
if not RUN_TAG and not PROMPT_TAG:
    RUN_NAME = LEGACY_RUN_NAMES.get((MODEL, SKELETON_STRATEGY, N_SAMPLES), RUN_NAME)

print(f"run: {RUN_NAME}  (cached examples are reused; set RUN_TAG for a fresh run)")

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=N_QUESTIONS,
    output_file=f"{RUN_NAME}.jsonl",
    pickle_dir=RUN_NAME,
)

run: tracelet_direct_n1_tp_react_gpt-4o  (cached examples are reused; set RUN_TAG for a fresh run)

[1/50] A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three   │
│ axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society │
│ in a Physics and Society article submitted to arXiv.org on August 11, 2016?                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2 memory messages


Thought: ──────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought:                                                                                                           
To solve this task, I need to first find the paper submitted to arXiv.org in June 2022 about AI regulation and     
identify the words on the axes of the figure. Then, I will find the Physics and Society article submitted in August
2016 and check if any of the axis words from the 2022 paper is mentioned there to describe a type of society.      
                                                                                                                   
I will start by searching for the paper about AI regulation submitted around June 2022 on arXiv.org, to find any   
figures with labeled axes.                                                                                         

...

65 memory messages


Thought: ──────────────────────────────────────────────────────────────────────────────────────────────────────────
Thank you for your patience. Let's ensure that the literal values are accurately replaced with sentinel            
identifiers.                                                                                                       
                                                                                                                   
Thought: I will replace the literal values `3455` and `0.04` with ARG0 and ARG1, respectively.                     

[Tracelet] 2 sentinel(s) found -- sampling 1 candidates.

[Step 23: Duration 3.35 seconds]

KeyboardInterrupt: 

In [7]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print(f"=== TraceletCodeAgent ({SKELETON_STRATEGY}, n_samples={N_SAMPLES}, {model_name}) ===")
print(f"Overall accuracy:   {correct}/{total} = {correct / total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg total tokens per question: {total_tokens:,.0f}")

print("\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== TraceletCodeAgent (post_process, n_samples=1, gpt-4o) ===
Overall accuracy:   13/50 = 26.0%
Avg time per question: 69.9s
Avg steps per question: 12.7
Avg total tokens per question: 215,349

Tool usage (total calls across all questions):
  web_search: 155
  visit_page: 84
  page_down: 81
  find_on_page_ctrl_f: 68
  final_answer: 46
  find_archived_url: 10
  inspect_file_as_text: 10
  find_next: 3
  visualizer: 2
